## 0. Libraries

In [1]:
import os
import re
import sys
import json
import glob
import time
import random
import logging
import tempfile
from getpass import getpass
from typing import Optional
from xml.etree import ElementTree as ET

import requests

## 1. Configuration
Set your API key, **contact email**, and paths here. Run the notebook from repo root.

In [ ]:
# ── Identity (NCBI asks for these so it can warn you before throttling) ──────
API_KEY = getpass("PubMed API key (blank = keyless, 3 req/s): ").strip() or None
TOOL    = "biomedical-literature-analysis"
EMAIL   = "EMAIL"          # <-- set to a REAL contact email

# Enforce a real contact email (NCBI policy; placeholder is rejected).
if "@" not in EMAIL or EMAIL.split("@")[-1].lower() in {"example.com", "example.org", "email.com"}:
    raise ValueError("Set EMAIL to a real contact address before running (NCBI requires it).")

# ── Output paths (relative to repo root) ────────────────────────────────────
RAW_DATA_DIR  = "data/0_raw/results"
PROGRESS_FILE = "data/0_raw/progress.json"
ERROR_DIR     = "data/0_raw/errors"
LOG_FILE      = "data/0_raw/harvest.log"

# ── Logging: console + file, so an unattended multi-day run keeps a record ──
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("harvest")

# ── Date range ──────────────────────────────────────────────────────────────
START_YEAR,  END_YEAR  = 1994, 2025
START_MONTH, END_MONTH = 1, 12

# ── Query — no disease pre-filtering; Journal Article[pt] drops editorials/letters ──
BASE_QUERY = (
    "english[Language]"
    " AND (USA[Affiliation] OR US[Affiliation])"
    " AND hasabstract[text]"
    " AND humans[Filter]"
    " AND Journal Article[pt]"
)

# ── Request settings ─────────────────────────────────────────────────────────
EFETCH_BATCH   = 200        # PMIDs per efetch POST (efetch retmax hard cap is 10000)
ESEARCH_RETMAX = 100_000    # esearch URL returns up to 100k UIDs per call
MAX_RETRIES    = 6          # network/HTTP retries per request (inside eutils_request)
CHUNK_RETRIES  = 3          # parse-error retries per efetch chunk (inside fetch_details)
TIMEOUT        = (10, 120)  # (connect, read) seconds
# Stay UNDER the rate ceiling (10/s with key, 3/s without):
MIN_INTERVAL   = 1.0 / 9.0 if API_KEY else 1.0 / 2.5